# Notebook 03 — Streaming y Manejo Robusto de Errores

**Semana:** 3  
**Bloques:** 3.3 Streaming · 3.4 Errores, rate limits, reintentos  
**Tiempo estimado:** 2.5-3 horas

## Qué vas a construir

- Streaming en tiempo real en OpenAI y Anthropic
- Acumulación del texto completo con métricas de latencia
- Un cliente LLM robusto con reintentos automáticos usando `tenacity`
- Simulación de errores sin gastar tokens
- Una clase `LLMClient` lista para producción que encapsula todo

## Prerrequisitos

```bash
pip install openai anthropic python-dotenv tenacity
```

---
## Celda 0 — Setup

In [ ]:
import os
import time
import random
import logging
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIConnectionError, APITimeoutError, BadRequestError
import anthropic
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    wait_random,
    retry_if_exception_type,
    before_sleep_log,
    RetryError
)

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

cliente_openai    = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), timeout=30.0)
cliente_anthropic = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"), timeout=30.0)

print("Setup completado.")

---
## Celda 1 — Streaming básico con OpenAI

Ejecuta esta celda y observa los tokens aparecer en tiempo real.
Mide también la latencia hasta el primer token (TTFT) y el tiempo total.

In [ ]:
PREGUNTA = "Explica en 4 párrafos cómo funciona la atención en los Transformers. Sé detallado."

print(f"Pregunta: {PREGUNTA}")
print("\n" + "="*60 + "\n")

t_inicio         = time.perf_counter()
t_primer_token   = None
texto_completo   = ""
n_chunks         = 0

stream = cliente_openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": PREGUNTA}],
    stream=True,
    stream_options={"include_usage": True}   # pide tokens en el último chunk
)

uso = None
for chunk in stream:
    delta = chunk.choices[0].delta.content if chunk.choices else None
    
    if delta:
        if t_primer_token is None:
            t_primer_token = time.perf_counter()    # primer token recibido
        texto_completo += delta
        n_chunks       += 1
        print(delta, end="", flush=True)
    
    if chunk.usage:       # último chunk
        uso = chunk.usage

t_fin = time.perf_counter()

# ── Métricas ──────────────────────────────────────────────────────────────────
ttft          = (t_primer_token - t_inicio) * 1000 if t_primer_token else 0
tiempo_total  = (t_fin - t_inicio) * 1000
tokens_output = uso.completion_tokens if uso else "N/A"
tps           = uso.completion_tokens / (t_fin - t_inicio) if uso else 0

print(f"\n\n{'='*60}")
print(f"Tiempo hasta primer token (TTFT): {ttft:.0f} ms")
print(f"Tiempo total de generación:       {tiempo_total:.0f} ms")
print(f"Chunks recibidos:                 {n_chunks}")
print(f"Tokens output:                    {tokens_output}")
print(f"Velocidad:                        {tps:.1f} tokens/s")
print(f"Longitud del texto:               {len(texto_completo)} caracteres")

**Observaciones** (edita esta celda):

- TTFT obtenido: ... ms — ¿es lo que esperabas?
- ¿Cuántos chunks llegaron? ¿Coincide con el número de tokens?
- Compara este tiempo con una llamada no-stream (ejecútala tú y mide): ...

---
## Celda 2 — Streaming con Anthropic

El SDK de Anthropic usa un context manager (`with ... as stream`).
`stream.text_stream` es el iterador de alto nivel que solo te da el texto.
Al salir del `with`, `stream.get_final_message()` tiene el mensaje completo con usage.

In [ ]:
print(f"Pregunta: {PREGUNTA}")
print("\n" + "="*60 + "\n")

t_inicio       = time.perf_counter()
t_primer_token = None
texto_completo = ""

with cliente_anthropic.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    messages=[{"role": "user", "content": PREGUNTA}]
) as stream:
    for texto in stream.text_stream:
        if t_primer_token is None:
            t_primer_token = time.perf_counter()
        texto_completo += texto
        print(texto, end="", flush=True)
    
    mensaje_final = stream.get_final_message()

t_fin = time.perf_counter()

ttft         = (t_primer_token - t_inicio) * 1000 if t_primer_token else 0
tiempo_total = (t_fin - t_inicio) * 1000
tps          = mensaje_final.usage.output_tokens / (t_fin - t_inicio)

print(f"\n\n{'='*60}")
print(f"Tiempo hasta primer token (TTFT): {ttft:.0f} ms")
print(f"Tiempo total:                     {tiempo_total:.0f} ms")
print(f"Tokens input / output:            {mensaje_final.usage.input_tokens} / {mensaje_final.usage.output_tokens}")
print(f"Velocidad:                        {tps:.1f} tokens/s")
print(f"Stop reason:                      {mensaje_final.stop_reason}")

---
## Celda 3 — Simular y manejar errores sin gastar tokens

Probar el manejo de errores contra la API real es difícil (necesitarías superar tu rate limit de verdad).
La solución: crea funciones que lancen las mismas excepciones que lanzaría la API real,
pero desde tu propio código.

Así puedes probar tu lógica de reintentos sin llamadas reales.

In [ ]:
from openai import InternalServerError
from unittest.mock import patch, MagicMock
import httpx

# ── Fábrica de errores falsos ────────────────────────────────────────────────
# Los constructores de las excepciones de OpenAI necesitan una respuesta HTTP.
# Creamos una respuesta mock mínima.

def crear_error_openai(tipo_error, mensaje="Error simulado"):
    """Crea una instancia de excepción de OpenAI con una respuesta HTTP falsa."""
    respuesta_falsa = httpx.Response(
        status_code=429 if "RateLimit" in tipo_error.__name__ else 500,
        request=httpx.Request("POST", "https://api.openai.com/v1/chat/completions")
    )
    return tipo_error(mensaje, response=respuesta_falsa, body={})


# ── Función que falla N veces antes de tener éxito ────────────────────────────
# Simula un servicio que está saturado y se recupera después de unos intentos.

class SimuladorFallos:
    """Simula un endpoint que falla `n_fallos` veces antes de responder bien."""
    
    def __init__(self, n_fallos: int, tipo_error=RateLimitError):
        self.n_fallos   = n_fallos
        self.tipo_error = tipo_error
        self.intentos   = 0
    
    def llamar(self, mensaje: str) -> str:
        self.intentos += 1
        print(f"  [Simulador] Intento {self.intentos}...")
        
        if self.intentos <= self.n_fallos:
            error = crear_error_openai(self.tipo_error, f"Error simulado #{self.intentos}")
            print(f"  [Simulador] Lanzando {self.tipo_error.__name__}")
            raise error
        
        # Éxito
        return f"Respuesta real tras {self.intentos} intentos: '{mensaje}'"


# ── Prueba sin reintentos — el error se propaga directamente ─────────────────
print("=== Sin reintentos ===")
sim = SimuladorFallos(n_fallos=2)
try:
    resultado = sim.llamar("Hola")
    print(f"Resultado: {resultado}")
except RateLimitError as e:
    print(f"Error sin capturar: {type(e).__name__} — la app se rompió")

---
## Celda 4 — Backoff exponencial con `tenacity`

Ahora añade reintentos automáticos. Observa cuánto espera entre cada intento.

In [ ]:
# ── Decorador de reintentos ───────────────────────────────────────────────────
# Ajusta estos parámetros y observa cómo cambia el comportamiento.

ERRORES_REINTENTABLES = (
    RateLimitError,
    APIConnectionError,
    APITimeoutError,
    InternalServerError,
    anthropic.RateLimitError,
    anthropic.APIConnectionError,
    anthropic.InternalServerError,
    anthropic.OverloadedError,
)

estrategia_reintento = retry(
    retry=retry_if_exception_type(ERRORES_REINTENTABLES),
    wait=wait_exponential(multiplier=1, min=1, max=10) + wait_random(0, 0.5),
    stop=stop_after_attempt(4),
    before_sleep=before_sleep_log(logger, logging.WARNING),
    reraise=True    # si se agotan los intentos, propaga la excepción original
)


@estrategia_reintento
def llamar_con_reintento(sim: SimuladorFallos, mensaje: str) -> str:
    """Envuelve cualquier llamada con la estrategia de reintento."""
    return sim.llamar(mensaje)


# ── Caso 1: falla 2 veces, luego éxito ───────────────────────────────────────
print("=== Caso 1: 2 fallos → éxito ===")
sim1 = SimuladorFallos(n_fallos=2)
t0   = time.perf_counter()
try:
    resultado = llamar_con_reintento(sim1, "Test")
    print(f"Resultado: {resultado}")
    print(f"Tiempo total con esperas: {(time.perf_counter() - t0):.2f}s")
except Exception as e:
    print(f"Falló definitivamente: {e}")

print()

# ── Caso 2: falla más veces de las que permite tenacity ──────────────────────
print("=== Caso 2: 5 fallos → se agotan los reintentos ===")
sim2 = SimuladorFallos(n_fallos=5)
t0   = time.perf_counter()
try:
    resultado = llamar_con_reintento(sim2, "Test")
    print(f"Resultado: {resultado}")
except RateLimitError as e:
    print(f"Fallido definitivamente tras {sim2.intentos} intentos")
    print(f"Tiempo total: {(time.perf_counter() - t0):.2f}s")

print()

# ── Caso 3: error NO reintentable (BadRequestError) ──────────────────────────
print("=== Caso 3: BadRequestError — no debe reintentar ===")
sim3 = SimuladorFallos(n_fallos=1, tipo_error=BadRequestError)
try:
    resultado = llamar_con_reintento(sim3, "Test")
except BadRequestError:
    print(f"BadRequestError capturado tras {sim3.intentos} intento(s) — correcto, no reintentó")
except Exception as e:
    print(f"Error inesperado: {type(e).__name__}")

**Observaciones** (edita esta celda):

- En el caso 1, ¿cuánto esperó entre el intento 1 y el 2? ¿Y entre el 2 y el 3?
- En el caso 3, ¿reintentó? ¿Por qué no?
- Si aumentas `min=1` a `min=5`, ¿qué cambia?

Prueba modificar los parámetros de `wait_exponential` y re-ejecuta.

---
## Celda 5 — Clase `LLMClient` lista para producción

Combina todo lo aprendido en las semanas 1, 2 y 3 en una clase cohesiva.
Esta será la base de todos tus proyectos del curso.

In [ ]:
from dataclasses import dataclass, field
from typing import Literal, Iterator, Callable


@dataclass
class LLMRespuesta:
    """Respuesta normalizada, independiente del proveedor."""
    texto:          str
    tokens_input:   int
    tokens_output:  int
    modelo:         str
    proveedor:      str
    latencia_ms:    float
    ttft_ms:        float | None = None

    @property
    def coste_usd(self) -> float:
        """Estimación de coste. Actualiza los precios si cambian."""
        precios = {
            "gpt-4o-mini":              {"input": 0.150, "output": 0.600},
            "gpt-4o":                   {"input": 2.500, "output": 10.00},
            "claude-haiku-4-5-20251001":{"input": 0.080, "output": 0.400},
            "claude-sonnet-4-6":        {"input": 3.000, "output": 15.00},
        }
        p = precios.get(self.modelo, {"input": 0, "output": 0})
        return (
            (self.tokens_input  / 1_000_000) * p["input"] +
            (self.tokens_output / 1_000_000) * p["output"]
        )


class LLMClient:
    """
    Cliente unificado para OpenAI y Anthropic.

    Características:
    - Streaming con métricas de latencia (TTFT y total)
    - Reintentos automáticos con backoff exponencial
    - Respuesta normalizada en LLMRespuesta
    - Logging integrado
    """

    MODELOS_DEFAULT = {
        "openai":    "gpt-4o-mini",
        "anthropic": "claude-haiku-4-5-20251001",
    }

    def __init__(
        self,
        proveedor: Literal["openai", "anthropic"] = "anthropic",
        modelo: str | None = None,
        timeout: float = 30.0,
        max_reintentos: int = 3,
    ):
        self.proveedor       = proveedor
        self.modelo          = modelo or self.MODELOS_DEFAULT[proveedor]
        self.max_reintentos  = max_reintentos
        self._logger         = logging.getLogger(self.__class__.__name__)

        if proveedor == "openai":
            self._cliente = OpenAI(
                api_key=os.getenv("OPENAI_API_KEY"),
                timeout=timeout
            )
        else:
            self._cliente = anthropic.Anthropic(
                api_key=os.getenv("ANTHROPIC_API_KEY"),
                timeout=timeout
            )

        # Construye el decorador de reintentos con los parámetros de la instancia
        self._reintento = retry(
            retry=retry_if_exception_type(ERRORES_REINTENTABLES),
            wait=wait_exponential(multiplier=1, min=2, max=60) + wait_random(0, 1),
            stop=stop_after_attempt(self.max_reintentos),
            before_sleep=before_sleep_log(self._logger, logging.WARNING),
            reraise=True
        )

    # ── Llamada estándar (sin streaming) ────────────────────────────────────

    def completar(
        self,
        mensajes: list[dict],
        system: str | None = None,
        temperature: float = 0.7,
        max_tokens: int = 1024,
    ) -> LLMRespuesta:
        """Llamada estándar con reintentos. Devuelve LLMRespuesta."""

        @self._reintento
        def _llamar():
            t0 = time.perf_counter()
            if self.proveedor == "openai":
                msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
                r = self._cliente.chat.completions.create(
                    model=self.modelo, messages=msgs,
                    temperature=temperature, max_tokens=max_tokens
                )
                return LLMRespuesta(
                    texto=r.choices[0].message.content,
                    tokens_input=r.usage.prompt_tokens,
                    tokens_output=r.usage.completion_tokens,
                    modelo=self.modelo, proveedor=self.proveedor,
                    latencia_ms=(time.perf_counter() - t0) * 1000
                )
            else:
                kwargs = dict(
                    model=self.modelo, messages=mensajes,
                    temperature=temperature, max_tokens=max_tokens
                )
                if system:
                    kwargs["system"] = system
                r = self._cliente.messages.create(**kwargs)
                return LLMRespuesta(
                    texto=r.content[0].text,
                    tokens_input=r.usage.input_tokens,
                    tokens_output=r.usage.output_tokens,
                    modelo=self.modelo, proveedor=self.proveedor,
                    latencia_ms=(time.perf_counter() - t0) * 1000
                )

        return _llamar()

    # ── Llamada con streaming ────────────────────────────────────────────────

    def completar_stream(
        self,
        mensajes: list[dict],
        system: str | None = None,
        max_tokens: int = 1024,
        on_token: Callable[[str], None] | None = None,
    ) -> LLMRespuesta:
        """
        Llamada con streaming.
        `on_token`: callback opcional llamado con cada token (p.ej. para imprimir).
        Devuelve LLMRespuesta con el texto completo al terminar.
        """
        t0             = time.perf_counter()
        t_primer_token = None
        texto_completo = ""

        if self.proveedor == "openai":
            msgs = ([{"role": "system", "content": system}] if system else []) + mensajes
            stream = self._cliente.chat.completions.create(
                model=self.modelo, messages=msgs, stream=True,
                max_tokens=max_tokens,
                stream_options={"include_usage": True}
            )
            uso = None
            for chunk in stream:
                delta = chunk.choices[0].delta.content if chunk.choices else None
                if delta:
                    if t_primer_token is None:
                        t_primer_token = time.perf_counter()
                    texto_completo += delta
                    if on_token:
                        on_token(delta)
                if chunk.usage:
                    uso = chunk.usage
            tokens_in  = uso.prompt_tokens     if uso else 0
            tokens_out = uso.completion_tokens if uso else 0

        else:  # anthropic
            kwargs = dict(model=self.modelo, messages=mensajes, max_tokens=max_tokens)
            if system:
                kwargs["system"] = system
            with self._cliente.messages.stream(**kwargs) as stream:
                for token in stream.text_stream:
                    if t_primer_token is None:
                        t_primer_token = time.perf_counter()
                    texto_completo += token
                    if on_token:
                        on_token(token)
                final = stream.get_final_message()
            tokens_in  = final.usage.input_tokens
            tokens_out = final.usage.output_tokens

        ttft = (t_primer_token - t0) * 1000 if t_primer_token else None
        return LLMRespuesta(
            texto=texto_completo,
            tokens_input=tokens_in,
            tokens_output=tokens_out,
            modelo=self.modelo,
            proveedor=self.proveedor,
            latencia_ms=(time.perf_counter() - t0) * 1000,
            ttft_ms=ttft
        )


# ── Prueba de la clase ────────────────────────────────────────────────────────

PREGUNTA_CORTA = "¿Cuál es la diferencia entre un proceso y un hilo en sistemas operativos?"

# Prueba 1: llamada estándar
print("=== Llamada estándar (Anthropic) ===")
llm = LLMClient(proveedor="anthropic")
resultado = llm.completar(
    mensajes=[{"role": "user", "content": PREGUNTA_CORTA}],
    system="Responde en máximo dos frases."
)
print(f"Texto:       {resultado.texto}")
print(f"Latencia:    {resultado.latencia_ms:.0f} ms")
print(f"Tokens:      {resultado.tokens_input} in / {resultado.tokens_output} out")
print(f"Coste:       ${resultado.coste_usd:.6f}")

print()

# Prueba 2: streaming con callback
print("=== Streaming (OpenAI) ===")
llm_oai = LLMClient(proveedor="openai")
resultado_stream = llm_oai.completar_stream(
    mensajes=[{"role": "user", "content": PREGUNTA_CORTA}],
    on_token=lambda t: print(t, end="", flush=True)
)
print(f"\n\nTTFT:        {resultado_stream.ttft_ms:.0f} ms")
print(f"Latencia:    {resultado_stream.latencia_ms:.0f} ms")
print(f"Coste:       ${resultado_stream.coste_usd:.6f}")

---
## Reto final — Chat de consola con streaming, reintentos y Ctrl+C limpio

Construye un chat de consola usando `LLMClient` que:
- Muestre tokens en streaming en tiempo real
- Imprima latencia y coste acumulado tras cada turno
- Al pulsar Ctrl+C, cierre limpiamente mostrando el coste total de la sesión
- Si hay un error irrecuperable (se agotan los reintentos), lo muestre sin romper el loop

El esqueleto ya gestiona Ctrl+C. Tu trabajo es completar los `# TODO`.

In [ ]:
def chat_streaming(
    proveedor: Literal["openai", "anthropic"] = "anthropic",
    system: str = "Eres un asistente útil. Responde siempre en español."
) -> None:
    """
    Chat de consola con streaming y métricas por turno.
    Escribe 'salir' para terminar, o pulsa Ctrl+C.
    """
    llm             = LLMClient(proveedor=proveedor)
    historial       = []
    coste_total     = 0.0
    n_turno         = 0

    print(f"Chat iniciado | {proveedor} / {llm.modelo}")
    print("Escribe 'salir' para terminar.\n")

    try:
        while True:
            # Lectura del input
            try:
                entrada = input("Tú: ").strip()
            except EOFError:
                break
            if not entrada:
                continue
            if entrada.lower() in ("salir", "exit", "quit"):
                break

            historial.append({"role": "user", "content": entrada})
            n_turno += 1

            print(f"\nAsistente: ", end="", flush=True)

            try:
                # TODO: llama a llm.completar_stream() con el historial completo.
                # Usa on_token para imprimir cada token en tiempo real.
                # Guarda el resultado en una variable `resultado`.
                resultado = None  # reemplaza esto

                # TODO: añade la respuesta al historial
                # historial.append(...)

                # TODO: acumula el coste y muestra las métricas del turno
                # coste_total += resultado.coste_usd
                # print(f"\n[turno {n_turno} | {resultado.latencia_ms:.0f}ms | ...")    

            except RetryError:
                print("\n[Error] No se pudo obtener respuesta tras varios intentos.")
            except Exception as e:
                print(f"\n[Error inesperado] {type(e).__name__}: {e}")

    except KeyboardInterrupt:
        pass  # Ctrl+C — cae al bloque finally

    finally:
        print(f"\n\n{'='*50}")
        print(f"Sesión terminada | {n_turno} turnos")
        print(f"Coste total: ${coste_total:.6f} USD")
        print(f"{'='*50}")


# Descomenta para ejecutar (mejor desde un .py en terminal que en Jupyter)
# chat_streaming(proveedor="anthropic")

### Extensiones opcionales

1. **Límite de contexto deslizante:** cuando el historial supera 20 mensajes, elimina los más antiguos manteniendo siempre el primer mensaje del sistema y los últimos N.

2. **Timeout visible:** si el primer token no llega en 3 segundos, muestra un indicador de `...` parpadeante mientras espera.

3. **Exporta la sesión:** al cerrar, guarda el historial completo en `sesion_YYYYMMDD_HHMMSS.json`.

In [ ]:
# Espacio para extensiones opcionales

---
## Autoevaluación de la semana

1. **¿Qué funcionó bien?**  
   ...

2. **¿Qué no entendiste del todo?**  
   ...

3. **¿Qué cambiarías la semana que viene?**  
   ...

4. **Tiempo real dedicado:**  
   ...

---
*Notebook 03 — Semana 3 del Plan de Formación AI Engineering*